# 4 — Neuron List

Compares universal circuits against token checker circuits to classify neurons
as concept-only, shared, or token-only.

Loops over all 9 epsilon x consistency settings in a single run.

In [1]:
# Cell 1 – Configuration
import subprocess, sys, os
for pkg in ["h5py", "numpy", "pandas"]:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=False)

import numpy as np
import pandas as pd
import h5py

LANG = "P"
MODEL_CONFIGS = {
    "QW": {"id": "Qwen/Qwen2.5-Coder-7B",                "n_layers": 28, "mlp_dim": 3584},
    "DS": {"id": "deepseek-ai/deepseek-coder-6.7b-base",  "n_layers": 32, "mlp_dim": 4096},
}

EPSILONS = [0.001, 0.1, 0.5]
CONSISTENCIES = [0.2, 0.5, 0.8]
MODELS_TO_RUN = ["QW", "DS"]
OBJECT_TYPE = "both"    # "ast", "builtin", or "both"

try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    import shutil
    mp = "/content/drive"
    subprocess.run(["fusermount", "-uz", mp], capture_output=True)
    if os.path.isdir(mp):
        shutil.rmtree(mp, ignore_errors=True)
    drive.mount(mp)
    DATA_DIR = "/content/drive/MyDrive/DATA/New-Atlas"
else:
    DATA_DIR = "/Users/piotrwilam/Data/New-Atlas"

os.makedirs(DATA_DIR, exist_ok=True)

print(f"Grid: {len(EPSILONS)} eps x {len(CONSISTENCIES)} cons = {len(EPSILONS)*len(CONSISTENCIES)} combos")
print(f"Type: {OBJECT_TYPE}, Models: {MODELS_TO_RUN}")

Grid: 3 eps x 3 cons = 9 combos
Type: both, Models: ['QW', 'DS']


In [2]:
# Cell 2 – Helper functions

def match_type(ds_name):
    if OBJECT_TYPE == "both":
        return True
    return ds_name.startswith(OBJECT_TYPE)


def load_universal_masks(obj_file, layers):
    """Load universal masks from HDF5."""
    universal = {}
    path = os.path.join(DATA_DIR, obj_file)
    if not os.path.exists(path):
        return None
    with h5py.File(path, "r") as f:
        um = f["universal_masks"]
        for lid in layers:
            lk = f"layer_{lid}"
            if lk not in um:
                continue
            for ds_name in um[lk]:
                if not match_type(ds_name):
                    continue
                if ds_name not in universal:
                    universal[ds_name] = {}
                universal[ds_name][lid] = np.array(um[lk][ds_name], dtype=bool)
    return universal


def load_checker_masks(chk_file, layers):
    """Load checker masks from HDF5."""
    checker = {}
    path = os.path.join(DATA_DIR, chk_file)
    if not os.path.exists(path):
        return None
    with h5py.File(path, "r") as f:
        tcm = f["token_checker_masks"]
        for lid in layers:
            lk = f"layer_{lid}"
            if lk not in tcm:
                continue
            for ds_name in tcm[lk]:
                if not match_type(ds_name):
                    continue
                if ds_name not in checker:
                    checker[ds_name] = {}
                checker[ds_name][lid] = np.array(tcm[lk][ds_name], dtype=bool)
    return checker


def detect_neuron_dim(masks):
    """Auto-detect neuron dimension from first mask."""
    for layers in masks.values():
        for mask in layers.values():
            return len(mask)
    return 4096  # safe fallback for largest model


print("Helpers defined.")

Helpers defined.


In [3]:
# Cell 3 – Loop over all models and settings, split output by layer halves

saved_files = []

for model_key in MODELS_TO_RUN:
    model_cfg = MODEL_CONFIGS[model_key]
    n_layers = model_cfg["n_layers"]
    prefix = f"{LANG}_{model_key}_"
    layers = list(range(n_layers))
    mid = n_layers // 2

    print(f"\n{'='*60}")
    print(f"Model: {model_key} ({n_layers} layers, split at {mid})")
    print(f"{'='*60}")

    for eps in EPSILONS:
        for cons in CONSISTENCIES:
            obj_file = f"{prefix}3_object_masks_eps{eps}_cons{cons}.h5"
            chk_file = f"{prefix}3_checker_masks_eps{eps}_cons{cons}.h5"

            print(f"\n--- {model_key} eps={eps}, cons={cons} ---")

            universal = load_universal_masks(obj_file, layers)
            if universal is None:
                print(f"  SKIP: {obj_file} not found")
                continue

            checker = load_checker_masks(chk_file, layers)
            if checker is None:
                print(f"  SKIP: {chk_file} not found")
                continue

            testable = sorted(set(universal.keys()) & set(checker.keys()))
            print(f"  Universal: {len(universal)}, Checker: {len(checker)}, Testable: {len(testable)}")

            if len(testable) == 0:
                print("  SKIP: no testable objects")
                continue

            neuron_dim = detect_neuron_dim(universal)

            rows = []
            for obj in testable:
                for lid in layers:
                    A = universal[obj].get(lid, np.zeros(neuron_dim, dtype=bool))
                    B = checker[obj].get(lid, np.zeros(neuron_dim, dtype=bool))

                    concept_only = np.where(A & ~B)[0].tolist()
                    both = np.where(A & B)[0].tolist()
                    token_only = np.where(~A & B)[0].tolist()

                    rows.append({
                        "object": obj,
                        "layer": lid,
                        "n_concept_only": len(concept_only),
                        "n_both": len(both),
                        "n_token_only": len(token_only),
                        "concept_only": str(concept_only),
                        "both": str(both),
                        "token_only": str(token_only),
                    })

            df = pd.DataFrame(rows)

            # Split and save by layer halves
            df_part1 = df[df["layer"] < mid]
            df_part2 = df[df["layer"] >= mid]

            for part_label, part_df in [("layers_part1", df_part1), ("layers_part2", df_part2)]:
                out_name = f"{prefix}4_neuron_list_eps{eps}_cons{cons}_{part_label}_{OBJECT_TYPE}.csv"
                out_path = os.path.join(DATA_DIR, out_name)
                part_df.to_csv(out_path, index=False)
                saved_files.append(out_name)

            print(f"  Saved: part1 ({len(df_part1)} rows, L0-{mid-1}) + part2 ({len(df_part2)} rows, L{mid}-{n_layers-1})")

print(f"\nDone: {len(saved_files)} files saved")


Model: QW (28 layers, split at 14)

--- QW eps=0.001, cons=0.2 ---
  Universal: 106, Checker: 58, Testable: 58
  Saved: part1 (812 rows, L0-13) + part2 (812 rows, L14-27)

--- QW eps=0.001, cons=0.5 ---
  Universal: 106, Checker: 58, Testable: 58
  Saved: part1 (812 rows, L0-13) + part2 (812 rows, L14-27)

--- QW eps=0.001, cons=0.8 ---
  Universal: 106, Checker: 58, Testable: 58
  Saved: part1 (812 rows, L0-13) + part2 (812 rows, L14-27)

--- QW eps=0.1, cons=0.2 ---
  Universal: 106, Checker: 58, Testable: 58
  Saved: part1 (812 rows, L0-13) + part2 (812 rows, L14-27)

--- QW eps=0.1, cons=0.5 ---
  Universal: 106, Checker: 58, Testable: 58
  Saved: part1 (812 rows, L0-13) + part2 (812 rows, L14-27)

--- QW eps=0.1, cons=0.8 ---
  Universal: 106, Checker: 58, Testable: 58
  Saved: part1 (812 rows, L0-13) + part2 (812 rows, L14-27)

--- QW eps=0.5, cons=0.2 ---
  Universal: 106, Checker: 58, Testable: 58
  Saved: part1 (812 rows, L0-13) + part2 (812 rows, L14-27)

--- QW eps=0.5, con

In [4]:
# Cell 4 – Summary

print("SAVED FILES:")
for f in saved_files:
    print(f"  {f}")

print(f"\n4 complete.")

try:
    from google.colab import runtime
    runtime.unassign()
except (ImportError, AttributeError):
    print("Not running on Colab -- skipping unassign.")

SAVED FILES:
  P_QW_4_neuron_list_eps0.001_cons0.2_layers_part1_both.csv
  P_QW_4_neuron_list_eps0.001_cons0.2_layers_part2_both.csv
  P_QW_4_neuron_list_eps0.001_cons0.5_layers_part1_both.csv
  P_QW_4_neuron_list_eps0.001_cons0.5_layers_part2_both.csv
  P_QW_4_neuron_list_eps0.001_cons0.8_layers_part1_both.csv
  P_QW_4_neuron_list_eps0.001_cons0.8_layers_part2_both.csv
  P_QW_4_neuron_list_eps0.1_cons0.2_layers_part1_both.csv
  P_QW_4_neuron_list_eps0.1_cons0.2_layers_part2_both.csv
  P_QW_4_neuron_list_eps0.1_cons0.5_layers_part1_both.csv
  P_QW_4_neuron_list_eps0.1_cons0.5_layers_part2_both.csv
  P_QW_4_neuron_list_eps0.1_cons0.8_layers_part1_both.csv
  P_QW_4_neuron_list_eps0.1_cons0.8_layers_part2_both.csv
  P_QW_4_neuron_list_eps0.5_cons0.2_layers_part1_both.csv
  P_QW_4_neuron_list_eps0.5_cons0.2_layers_part2_both.csv
  P_QW_4_neuron_list_eps0.5_cons0.5_layers_part1_both.csv
  P_QW_4_neuron_list_eps0.5_cons0.5_layers_part2_both.csv
  P_QW_4_neuron_list_eps0.5_cons0.8_layers_part